# Lesson 8.2: What Is Query Rewriting and How Does It Help Retrieval?

**Companion notebook for Lesson 8.2**

---

| Section | What you will build |
|---|---|
| 1. The Mismatch Problem | Measure the vocabulary gap with cosine similarity |
| 2. Technical Corpus + Retriever | 10-doc knowledge base; baseline retrieval failures |
| 3. Sub-query Decomposition | Split complex → simple; merge results |
| 4. HyDE | Hypothetical answer as the search vector; PCA visualisation |
| 5. Step-back Prompting | Broaden hyper-specific queries before retrieval |
| 6. Decision Logic | `classify_query()` + combined `smart_retrieve()` pipeline |
| 7. Recall@k Evaluation | Measure improvement across all strategies |
| 8. Claude API | Real LLM rewriting with Claude |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`, `scikit-learn`  
**Optional (Section 8):** `anthropic` — enables real Claude API calls

In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib scikit-learn
# !pip install anthropic   # optional — for real LLM rewriting in Section 8

In [ ]:
%matplotlib inline

import os
import re
import warnings

os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

def show_plot():
    plt.tight_layout()
    plt.show()

try:
    from sklearn.decomposition import PCA
    SKLEARN_AVAILABLE = True
except ImportError:
    SKLEARN_AVAILABLE = False
    print('sklearn not installed — PCA visualisation in Section 4 will be skipped.')
    print('Install with: pip install scikit-learn')

print('Imports ready.')

---
## 1. The Query–Document Mismatch Problem

The surprise pattern almost every AI engineering team hits in production:

```
┌──────────────────────────────────────────────────────────────────────┐
│                THE QUERY-DOCUMENT MISMATCH                           │
└──────────────────────────────────────────────────────────────────────┘

  USER WRITES:                DOCUMENT SAYS:

  "why is my app slow??"      "Performance degradation in client applications
                               is typically attributable to inefficient database
                               query patterns, particularly N+1 lookups and
                               unindexed joins…"

       ↓ embed                        ↓ embed

  [query vector]              [document vector]

       ←─────────── far apart in vector space ──────────────→

  Why? Three mismatches:
  1. REGISTER:    casual vs. formal
  2. SPECIFICITY: symptom vs. cause
  3. VOCABULARY:  almost no shared words

  Solution: don't search with the user's raw query — rewrite it first.
```

In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading embedding model (all-MiniLM-L6-v2)...')
embedder    = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('Embedder ready.')

# The blog's canonical example
casual_query = 'why is my app slow??'
formal_doc   = (
    'Performance degradation in client applications is typically attributable '
    'to inefficient database query patterns, particularly N+1 lookups and '
    'unindexed joins. Profiling tools such as pg_stat_statements for PostgreSQL '
    'can identify slow query candidates.'
)
hyde_answer  = (
    'Application performance degradation is often attributable to inefficient '
    'database query patterns, particularly N+1 lookups where each record triggers '
    'a separate database call rather than a single batched join. Unindexed joins '
    'further compound the problem. Profiling the database can identify root causes.'
)

emb_query  = embedder.encode(casual_query,  convert_to_tensor=True, show_progress_bar=False)
emb_doc    = embedder.encode(formal_doc,    convert_to_tensor=True, show_progress_bar=False)
emb_hyde   = embedder.encode(hyde_answer,   convert_to_tensor=True, show_progress_bar=False)

sim_raw    = float(util.cos_sim(emb_query, emb_doc)[0][0])
sim_hyde   = float(util.cos_sim(emb_hyde,  emb_doc)[0][0])

print(f'\nQuery:     "{casual_query}"')
print(f'Document:  "{formal_doc[:80]}..."')
print()
print(f'Cosine similarity (raw query  → doc): {sim_raw:.3f}')
print(f'Cosine similarity (HyDE answer → doc): {sim_hyde:.3f}')
print(f'Improvement: {(sim_hyde - sim_raw):.3f}  ({(sim_hyde/sim_raw - 1)*100:.0f}% lift)')
print()
print('Insight: the HyDE hypothetical answer shares vocabulary with the real document.')
print('It uses "N+1 lookups", "database query patterns", "performance degradation" —')
print('exactly the terms in the document. The raw query uses none of them.')

In [ ]:
# Visualise the similarity gap
fig, ax = plt.subplots(figsize=(8, 4))

labels  = ['Raw query\n"why is my app slow??"', 'HyDE hypothetical answer\n(formal, technical language)']
values  = [sim_raw, sim_hyde]
colors  = ['#E53935', '#2E7D32']

bars = ax.barh(labels, values, color=colors, alpha=0.85, height=0.4)
ax.axvline(0.5, color='#F57F17', linewidth=2, linestyle='--', label='Threshold for reliable retrieval')
ax.set_xlim(0, 1)
ax.set_xlabel('Cosine Similarity to Formal Document')
ax.set_title('Query–Document Cosine Similarity: Raw vs. HyDE', fontweight='bold')
ax.legend(fontsize=10)

for bar, val in zip(bars, values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

show_plot()

print('The HyDE answer does not need to be correct.')
print('It just needs to sound like the kind of document that would answer the question.')
print('Even a wrong hypothetical answer uses the right vocabulary — and that is enough.')

---
## 2. Technical Corpus + Baseline Retriever

Ten documents written in formal technical language — the kind produced by engineers writing internal docs, API references, and runbooks. Users will query this corpus with casual, vague, symptom-oriented language.

In [ ]:
CORPUS = [
    # ── Performance ──────────────────────────────────────────────────────────
    {'id': 0, 'title': 'Performance Degradation: Root Cause Analysis',
     'text': 'Performance degradation in client applications is typically attributable '
             'to inefficient database query patterns, particularly N+1 lookups and '
             'unindexed joins. Profiling tools such as pg_stat_statements for PostgreSQL '
             'can identify slow query candidates.'},

    {'id': 1, 'title': 'Memory Leak Detection and Mitigation',
     'text': 'Heap memory exhaustion in long-running server processes manifests as '
             'gradually increasing response latency over time. Memory profiling tools '
             'such as Valgrind and heaptrack enable identification of unreleased '
             'allocations in native code.'},

    {'id': 2, 'title': 'Read-Heavy Workload Optimisation via Caching',
     'text': 'Implementing a read-through cache layer reduces database egress for '
             'frequently accessed, temporally stable data. Cache invalidation strategies '
             'include TTL-based expiration and event-driven invalidation via pub/sub '
             'mechanisms, each suited to different data volatility profiles.'},

    # ── Authentication ────────────────────────────────────────────────────────
    {'id': 3, 'title': 'OAuth 2.0 Authorization Code Grant Flow',
     'text': 'The authorization code grant type involves a redirection-based flow where '
             'the client application obtains an authorization code from the authorization '
             'server, subsequently exchanging it for an access token via a back-channel '
             'request. This flow is suitable for server-side applications that can '
             'maintain client secret confidentiality.'},

    {'id': 4, 'title': 'JWT Security: Algorithm Selection and Validation',
     'text': 'JWT signature verification must be performed using asymmetric cryptographic '
             'algorithms (RS256 or ES256) in production environments. The "none" algorithm '
             'must be explicitly disabled to prevent algorithm confusion attacks. Token '
             'expiry (exp claim) and audience (aud claim) validation are mandatory.'},

    # ── API ───────────────────────────────────────────────────────────────────
    {'id': 5, 'title': 'API Rate Limiting: Token Bucket and Sliding Window',
     'text': 'Token bucket and sliding window counter algorithms are the predominant '
             'approaches for API rate limiting. The token bucket algorithm provides '
             'burst tolerance while maintaining average throughput constraints. Sliding '
             'window counters offer more precise rate enforcement at higher storage cost.'},

    {'id': 6, 'title': 'Webhook Delivery Guarantees and Retry Semantics',
     'text': 'Webhook delivery employs exponential backoff with jitter to mitigate '
             'thundering herd effects during downstream service recovery. At-least-once '
             'delivery semantics necessitate idempotency keys on the receiving endpoint '
             'to prevent duplicate event processing.'},

    # ── Data pipelines ────────────────────────────────────────────────────────
    {'id': 7, 'title': 'Streaming Pipeline Architecture and Backpressure',
     'text': 'Streaming ingestion architectures utilising Apache Kafka or Kinesis Data '
             'Streams provide sub-second latency with horizontal scalability. Backpressure '
             'handling is critical to prevent producer saturation during consumer lag '
             'events caused by downstream processing bottlenecks.'},

    {'id': 8, 'title': 'Eventual Consistency and CRDTs in Distributed Systems',
     'text': 'Eventual consistency models trade linearizability for availability, as '
             'formalised by the CAP theorem. Vector clocks and CRDTs (Conflict-free '
             'Replicated Data Types) enable convergent state reconciliation across '
             'distributed replicas without central coordination.'},

    # ── Observability ─────────────────────────────────────────────────────────
    {'id': 9, 'title': 'Distributed Tracing with OpenTelemetry',
     'text': 'Distributed tracing with OpenTelemetry provides causal propagation of '
             'request context across service boundaries. Tail-based sampling enables '
             'anomaly-conditional retention of request traces, unlike head-based '
             'sampling which makes the retention decision before the trace completes.'},
]

print(f'Corpus: {len(CORPUS)} documents')
for doc in CORPUS:
    print(f'  [{doc["id"]}] {doc["title"]}')

In [ ]:
corpus_texts  = [doc['text'] for doc in CORPUS]
corpus_embeds = embedder.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=False)


def retrieve(query: str, top_k: int = 3):
    """Return top_k (doc, score) pairs by cosine similarity."""
    q_emb   = embedder.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores  = util.cos_sim(q_emb, corpus_embeds)[0].cpu().numpy()
    indices = np.argsort(scores)[::-1][:top_k]
    return [(CORPUS[i], float(scores[i])) for i in indices]


print(f'Embedded {len(CORPUS)} docs (dim={corpus_embeds.shape[1]})')

# Show the mismatch in action — casual queries against formal docs
CASUAL_QUERIES = [
    ('why is my app so slow?',                               0),  # → Performance
    ('my server keeps running out of memory',                1),  # → Memory leak
    ('how do i add google login to my app?',                 3),  # → OAuth
    ('my api keeps getting throttled',                       5),  # → Rate limiting
    ('my webhooks are failing and dropping messages',        6),  # → Webhooks
    ('my database shows different data on different servers',8),  # → Consistency
]

print(f'\n{"Casual query":<52} {"Top-1 retrieved":<38} {"Relevant?"}')
print('-' * 100)
for query, relevant_id in CASUAL_QUERIES:
    top = retrieve(query, top_k=1)
    top_doc, top_score = top[0]
    hit = '✅' if top_doc['id'] == relevant_id else f'❌ (want doc {relevant_id})'
    print(f'{query:<52} [{top_doc["id"]}] {top_doc["title"][:32]:<38} {hit}')
print()
print('❌ rows = the retriever found the wrong document.')
print('The casual query and the relevant formal doc share too little vocabulary.')

---
## 3. Sub-query Decomposition

Complex questions cannot be answered in a single retrieval hop — they contain multiple distinct information needs that should each be retrieved separately.

```
┌────────────────────────────────────────────────────────────────────────┐
│                    SUB-QUERY DECOMPOSITION                             │
└────────────────────────────────────────────────────────────────────────┘

  Complex query
  "My app is slow under load AND our login keeps getting bypassed"
         │
         ▼  Planner LLM
  ┌──────┴──────────────────────────────────────────────────────┐
  │ 1. "What causes performance degradation under concurrent     │
  │     read load?"                                             │
  │ 2. "What caching strategies reduce database load?"          │
  │ 3. "How can authentication bypass vulnerabilities be        │
  │     prevented?"                                             │
  └──────┬──────────────────────────────────────────────────────┘
         │  Retrieve each independently
         ▼
  Sub-query 1 → [doc 0, doc 2]   (performance docs)
  Sub-query 2 → [doc 2, doc 0]   (caching doc)
  Sub-query 3 → [doc 3, doc 4]   (auth docs)
         │
         ▼  Merge + deduplicate (RRF or max-score)
  Final context: [doc 0, doc 2, doc 3, doc 4] → LLM synthesis
```

In [ ]:
class MockDecomposeLLM:
    """
    Simulates a planner LLM that breaks complex questions into focused sub-queries.
    In production, replace with a real LLM call (see Section 8).
    """

    _DECOMPOSITIONS = {
        # (keyword_signature): [sub_query_1, sub_query_2, ...]
        'slow load auth':
            ['What causes performance degradation in applications under concurrent read load?',
             'What caching strategies reduce database load for high-traffic read workloads?',
             'How can authentication bypass vulnerabilities be prevented in web applications?'],

        'slow login':
            ['What causes application performance degradation?',
             'How is secure user authentication implemented?'],

        'memory webhook':
            ['What causes memory exhaustion in long-running server processes?',
             'How does webhook retry logic handle delivery failures?'],

        'data pipeline slow':
            ['What causes streaming data pipeline lag and consumer backlog?',
             'What causes application performance degradation?'],
    }

    def decompose(self, query: str) -> list:
        q = query.lower()
        for key, subs in self._DECOMPOSITIONS.items():
            if all(kw in q for kw in key.split()):
                return subs
        # Fallback: split on conjunctions
        parts = re.split(r'\band\b|\balso\b|\bplus\b', q, flags=re.IGNORECASE)
        parts = [p.strip() for p in parts if len(p.strip()) > 10]
        return parts if len(parts) > 1 else [query]


decompose_llm = MockDecomposeLLM()


def decompose_retrieve(query: str, top_k_per_sub: int = 2) -> tuple:
    """
    Decompose → retrieve for each sub-query → merge with max-score dedup.
    Returns (merged_results, sub_queries).
    """
    sub_queries = decompose_llm.decompose(query)
    merged = {}  # doc_id → (doc, best_score)
    for sq in sub_queries:
        for doc, score in retrieve(sq, top_k=top_k_per_sub):
            if doc['id'] not in merged or score > merged[doc['id']][1]:
                merged[doc['id']] = (doc, score)
    results = sorted(merged.values(), key=lambda x: x[1], reverse=True)
    return results, sub_queries


# Demo: a complex query the retriever can't handle in one shot
complex_query = (
    'my app is slow under load '
    'and our auth login keeps getting bypassed'
)

print(f'Complex query: "{complex_query}"')
print()

print('--- Baseline (raw query, top-3) ---')
for doc, score in retrieve(complex_query, top_k=3):
    print(f'  [{doc["id"]}] {doc["title"]} (score={score:.3f})')

print()
results, sub_qs = decompose_retrieve(complex_query)
print(f'--- After decomposition ({len(sub_qs)} sub-queries) ---')
for sq in sub_qs:
    print(f'  ↳ "{sq}"')
print()
print('Merged results (sorted by max sub-query score):')
for doc, score in results[:6]:
    print(f'  [{doc["id"]}] {doc["title"]} (score={score:.3f})')
print()
print('Decomposition surfaces docs 0, 2, 3, 4 — covering both the performance')
print('and authentication aspects of the original compound question.')

In [ ]:
# Visualise which documents each sub-query retrieves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: baseline top-5 scores for complex query
base_results = retrieve(complex_query, top_k=len(CORPUS))
base_doc_ids = [d['id'] for d, _ in base_results]
base_scores  = [s for _, s in base_results]
target_ids   = {0, 2, 3, 4}  # relevant docs for this compound question

base_colors = ['#2E7D32' if CORPUS[base_doc_ids[i]]['id'] in target_ids
               else '#90CAF9' for i in range(len(base_doc_ids))]
x_labels = [f'[{did}]' for did in base_doc_ids]

axes[0].bar(x_labels[:7], base_scores[:7], color=base_colors[:7], alpha=0.85)
axes[0].set_xlabel('Document ID (left = highest scored)')
axes[0].set_ylabel('Cosine Similarity')
axes[0].set_title('Baseline: Raw Complex Query\n(one retrieval call)', fontweight='bold')
for i, (did, score) in enumerate(zip(base_doc_ids[:7], base_scores[:7])):
    axes[0].text(i, score + 0.005, f'{score:.2f}', ha='center', fontsize=8)

# Right: decomposed — show which docs each sub-query pulls
sq_colors = ['#1565C0', '#6A1B9A', '#BF360C']
_, sub_qs = decompose_retrieve(complex_query)

x     = np.arange(len(CORPUS))
width = 0.25

for j, sq in enumerate(sub_qs[:3]):
    sq_results = retrieve(sq, top_k=len(CORPUS))
    sq_scores  = [0.0] * len(CORPUS)
    for doc, score in sq_results:
        sq_scores[doc['id']] = score
    axes[1].bar(x + j * width, sq_scores, width,
                color=sq_colors[j], alpha=0.75,
                label=f'Sub-query {j+1}: "{sq[:35]}..."')

axes[1].set_xticks(x + width)
axes[1].set_xticklabels([f'[{i}]' for i in range(len(CORPUS))], fontsize=9)
axes[1].set_xlabel('Document ID')
axes[1].set_ylabel('Cosine Similarity')
axes[1].set_title('Decomposition: Each Sub-query Finds Different Relevant Docs',
                   fontweight='bold')
axes[1].legend(fontsize=8, loc='upper right')

legend_handles = [
    mpatches.Patch(color='#2E7D32', label='Relevant docs (target ids: 0, 2, 3, 4)'),
    mpatches.Patch(color='#90CAF9', label='Irrelevant docs'),
]
axes[0].legend(handles=legend_handles, fontsize=9)

show_plot()

---
## 4. HyDE: Hypothetical Document Embeddings

Instead of embedding the user's question, embed a **hypothetical answer** to the question — even if that answer is made up.

```
┌────────────────────────────────────────────────────────────────────────┐
│                  WHY HyDE WORKS                                        │
└────────────────────────────────────────────────────────────────────────┘

  Vector space (simplified 2D):

  ·  ·  ·  ·  ·  ·  ·  ·         ← real documents cluster here
        doc0  doc1  doc2           (formal, technical vocabulary)
             ★
             HyDE answer           ← hypothetical answer also clusters here
             (even if wrong,       (because it uses the same vocabulary)
             uses same vocab)




  ✕ raw user query               ← user query lives down here
  "why is my app slow??"         (casual, symptom-oriented vocabulary)

  Key insight: a wrong hypothetical answer still uses the right vocabulary.
  That is enough to dramatically reduce the embedding distance to real docs.
```

In [ ]:
class MockHydeLLM:
    """
    Simulates a generative LLM producing hypothetical document snippets.

    Key design principle: the hypothetical answer does NOT need to be correct.
    It just needs to sound like the relevant document — same vocabulary,
    same register, same level of specificity.

    Compare each entry below to its matching corpus document (same words!).
    """

    _HYPOTHETICALS = {
        'slow':
            'Application performance degradation is often attributable to inefficient '
            'database query patterns, particularly N+1 lookups where each record '
            'triggers a separate database call rather than a batched join. Unindexed '
            'joins further compound latency. Profiling tools can identify the root '
            'cause of slow queries.',

        'memory':
            'Heap memory exhaustion in server processes typically results from '
            'unreleased allocations that accumulate in long-running processes. '
            'Memory profiling tools identify which allocations are not freed, '
            'enabling mitigation of the underlying memory leak.',

        'reads':
            'Optimising read performance under high concurrent load involves a '
            'read-through cache layer that reduces database egress for frequently '
            'accessed data. TTL-based cache invalidation and pub/sub event-driven '
            'strategies suit different data volatility profiles.',

        'login google':
            'Adding Google login requires implementing the OAuth 2.0 authorization '
            'code grant flow. The client obtains an authorization code via redirection '
            'to the authorization server, then exchanges it for an access token via '
            'a secure back-channel request.',

        'token safe':
            'JWT token security requires asymmetric algorithm selection (RS256 or '
            'ES256) for signature verification. The none algorithm must be disabled '
            'to prevent algorithm confusion attacks. Token expiry and audience claim '
            'validation are mandatory in production.',

        'throttled':
            'API rate limiting uses token bucket or sliding window algorithms to '
            'control throughput. The token bucket algorithm provides burst tolerance '
            'while enforcing average throughput constraints, preventing backend '
            'saturation under spike traffic.',

        'webhook':
            'Webhook delivery failures are handled by exponential backoff retry '
            'mechanisms with jitter to prevent thundering herd effects. Idempotency '
            'keys on the receiving endpoint enable at-least-once delivery semantics '
            'without duplicate processing.',

        'pipeline':
            'Streaming data pipeline lag occurs when consumer throughput falls below '
            'the producer ingestion rate. Kafka-based architectures with backpressure '
            'handling prevent producer saturation during consumer lag events caused '
            'by downstream processing bottlenecks.',

        'different data':
            'Inconsistent data across distributed database replicas is a property of '
            'eventual consistency models that trade linearizability for availability. '
            'CRDTs and vector clocks enable convergent state reconciliation across '
            'replicas without central coordination.',

        'services':
            'Visibility into microservice behaviour requires distributed tracing to '
            'propagate request context across service boundaries. OpenTelemetry with '
            'tail-based sampling enables anomaly-conditional retention of request '
            'traces for debugging.',
    }

    def hypothesize(self, query: str) -> str:
        q = query.lower()
        for key, hypothesis in self._HYPOTHETICALS.items():
            if all(kw in q for kw in key.split()):
                return hypothesis
        # Generic fallback: still uses technical register
        return (
            f'The technical documentation for this topic typically describes '
            f'the underlying mechanisms, failure modes, and recommended '
            f'mitigation strategies using industry-standard tooling and approaches.'
        )


hyde_llm = MockHydeLLM()
print('MockHydeLLM ready.')
print()
print('Key design: hypothetical answers copy the vocabulary of formal docs.')
print('Example for query: my server keeps running out of memory')
hyp = hyde_llm.hypothesize('my server keeps running out of memory')
print(f'  Hypothetical: "{hyp[:100]}..."')
print()
print('Real doc 1: "Heap memory exhaustion in long-running server processes manifests...')

In [ ]:
def hyde_retrieve(query: str, top_k: int = 3) -> tuple:
    """
    Generate a hypothetical answer, embed it instead of the query,
    then retrieve by cosine similarity to that hypothetical embedding.

    Returns (results, hypothesis_text).
    """
    hypothesis = hyde_llm.hypothesize(query)
    hyp_emb    = embedder.encode(hypothesis, convert_to_tensor=True, show_progress_bar=False)
    scores     = util.cos_sim(hyp_emb, corpus_embeds)[0].cpu().numpy()
    indices    = np.argsort(scores)[::-1][:top_k]
    results    = [(CORPUS[i], float(scores[i])) for i in indices]
    return results, hypothesis


# Demo: casual queries that fail with raw retrieval, succeed with HyDE
hyde_demos = [
    ('why is my app so slow?',                 0),
    ('my server keeps running out of memory',  1),
    ('my api keeps getting throttled',         5),
]

print('=== HyDE vs Baseline — Side-by-Side ===\n')
for query, relevant_id in hyde_demos:
    base_results             = retrieve(query, top_k=3)
    hyde_results, hypothesis = hyde_retrieve(query, top_k=3)

    base_top_id = base_results[0][0]['id']
    hyde_top_id = hyde_results[0][0]['id']

    base_rel_score = next((s for d, s in base_results if d['id'] == relevant_id), 0.0)
    hyde_rel_score = next((s for d, s in hyde_results if d['id'] == relevant_id), 0.0)

    print(f'Q: "{query}"')
    print(f'   Relevant doc: [{relevant_id}] {CORPUS[relevant_id]["title"]}')
    print(f'   Baseline → top-1: [{base_top_id}]  |  sim to relevant: {base_rel_score:.3f}')
    print(f'   HyDE     → top-1: [{hyde_top_id}]  |  sim to relevant: {hyde_rel_score:.3f}  '
          f'(+{hyde_rel_score - base_rel_score:.3f})')
    print(f'   Hypothesis: "{hypothesis[:80]}..."')
    print()

In [ ]:
# Similarity comparison bar chart: raw query vs. HyDE across all docs
demo_q       = 'why is my app so slow?'
demo_rel_id  = 0

q_emb_raw     = embedder.encode(demo_q, convert_to_tensor=True, show_progress_bar=False)
_, demo_hyp   = hyde_retrieve(demo_q)
q_emb_hyde    = embedder.encode(demo_hyp, convert_to_tensor=True, show_progress_bar=False)

raw_sims  = util.cos_sim(q_emb_raw,  corpus_embeds)[0].cpu().numpy()
hyde_sims = util.cos_sim(q_emb_hyde, corpus_embeds)[0].cpu().numpy()

fig, ax = plt.subplots(figsize=(14, 5))
x     = np.arange(len(CORPUS))
width = 0.35

bars_raw  = ax.bar(x - width/2, raw_sims,  width, color='#E53935', alpha=0.75, label='Raw query')
bars_hyde = ax.bar(x + width/2, hyde_sims, width, color='#2E7D32', alpha=0.75, label='HyDE (hypothetical answer)')

# Highlight the relevant document
ax.axvspan(demo_rel_id - 0.5, demo_rel_id + 0.5, alpha=0.12, color='#F57F17',
           label=f'Relevant doc [{demo_rel_id}]')
ax.set_xticks(x)
ax.set_xticklabels(
    [f'[{i}]\n{CORPUS[i]["title"][:18]}...' for i in range(len(CORPUS))],
    fontsize=8, rotation=15, ha='right'
)
ax.set_ylabel('Cosine Similarity')
ax.set_title(f'Query: "{demo_q}"\nCosine Similarity to Each Document: Raw vs. HyDE',
              fontweight='bold')
ax.legend(fontsize=10)

# Annotate the relevant doc's scores
ax.annotate(f'Raw: {raw_sims[demo_rel_id]:.3f}',
            (demo_rel_id - width/2, raw_sims[demo_rel_id]),
            xytext=(demo_rel_id - 1.5, raw_sims[demo_rel_id] + 0.06),
            arrowprops=dict(arrowstyle='->', color='#E53935'),
            color='#E53935', fontsize=10, fontweight='bold')
ax.annotate(f'HyDE: {hyde_sims[demo_rel_id]:.3f}',
            (demo_rel_id + width/2, hyde_sims[demo_rel_id]),
            xytext=(demo_rel_id + 0.8, hyde_sims[demo_rel_id] + 0.06),
            arrowprops=dict(arrowstyle='->', color='#2E7D32'),
            color='#2E7D32', fontsize=10, fontweight='bold')

show_plot()

print(f'Relevant doc [{demo_rel_id}]: "{CORPUS[demo_rel_id]["title"]}"')
print(f'  Raw query similarity:  {raw_sims[demo_rel_id]:.3f}')
print(f'  HyDE similarity:       {hyde_sims[demo_rel_id]:.3f}  (+{hyde_sims[demo_rel_id]-raw_sims[demo_rel_id]:.3f})')
print()
print('HyDE lifts the relevant doc without inflating all other docs equally.')
print('The vocabulary match is specific — it targets the right neighbourhood.')

In [ ]:
# PCA visualisation: where does the raw query land vs. the HyDE answer?
if not SKLEARN_AVAILABLE:
    print('sklearn not available — install with: pip install scikit-learn')
else:
    all_vecs   = np.vstack([
        corpus_embeds.cpu().numpy(),
        q_emb_raw.cpu().numpy().reshape(1, -1),
        q_emb_hyde.cpu().numpy().reshape(1, -1),
    ])
    pca    = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(all_vecs)

    doc_coords  = coords[:len(CORPUS)]
    raw_coord   = coords[len(CORPUS)]
    hyde_coord  = coords[len(CORPUS) + 1]

    # Colour-code docs by category
    TOPIC_COLORS = {
        (0, 1, 2): ('#E53935', 'Performance'),
        (3, 4):    ('#1565C0', 'Auth'),
        (5, 6):    ('#6A1B9A', 'API'),
        (7, 8):    ('#2E7D32', 'Data'),
        (9,):      ('#F57F17', 'Observability'),
    }
    doc_colors = {}
    for ids, (color, _) in TOPIC_COLORS.items():
        for i in ids:
            doc_colors[i] = color

    fig, ax = plt.subplots(figsize=(11, 8))

    # Plot corpus documents
    for doc_id, (x, y) in enumerate(doc_coords):
        color = doc_colors.get(doc_id, '#888')
        ax.scatter(x, y, color=color, s=160, zorder=3, alpha=0.85)
        ax.annotate(f'[{doc_id}]', (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=9)

    # Plot raw query
    ax.scatter(*raw_coord, color='black', marker='X', s=300, zorder=5,
                label=f'Raw query: "{demo_q}"')
    ax.annotate('Raw query\n✕', raw_coord, textcoords='offset points',
                xytext=(8, -12), fontsize=9, color='black', fontweight='bold')

    # Plot HyDE answer
    ax.scatter(*hyde_coord, color='gold', marker='*', s=500, zorder=5,
                label='HyDE (hypothetical answer)', edgecolors='darkorange', linewidths=1.5)
    ax.annotate('HyDE answer\n★', hyde_coord, textcoords='offset points',
                xytext=(8, 6), fontsize=9, color='darkorange', fontweight='bold')

    # Draw lines from query to relevant doc [0]
    rel = doc_coords[demo_rel_id]
    ax.annotate('', xy=rel, xytext=raw_coord,
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5, linestyle='dashed'))
    ax.annotate('', xy=rel, xytext=hyde_coord,
                arrowprops=dict(arrowstyle='->', color='darkorange', lw=2.0))

    # Legend for topic colours
    for ids, (color, label) in TOPIC_COLORS.items():
        ax.scatter([], [], color=color, s=80, label=f'Topic: {label}')

    ax.set_title(
        'PCA Projection (384D → 2D)\n'
        'HyDE answer ★ lands closer to relevant docs than raw query ✕',
        fontweight='bold'
    )
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%} variance)')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%} variance)')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(True, alpha=0.3)
    show_plot()

    # Compute actual 384D distances
    doc0_vec   = corpus_embeds[demo_rel_id].cpu().numpy()
    dist_raw   = float(np.linalg.norm(q_emb_raw.cpu().numpy()  - doc0_vec))
    dist_hyde  = float(np.linalg.norm(q_emb_hyde.cpu().numpy() - doc0_vec))
    print(f'Euclidean distance to doc [{demo_rel_id}] in 384D space:')
    print(f'  Raw query:  {dist_raw:.4f}')
    print(f'  HyDE:       {dist_hyde:.4f}  ({(1 - dist_hyde/dist_raw)*100:.0f}% closer)')
    print()
    print('Note: PCA loses most variance when projecting 384D → 2D.')
    print('The 384D cosine similarities (previous cell) are the authoritative signal.')

---
## 5. Step-back Prompting: Zoom Out Before Zooming In

When a query is too specific, the answer may live in a broader summary document — not in a document about the exact term the user mentioned.

| When to use step-back | Example |
|---|---|
| Very specific algorithm / entity name | "does the token bucket algorithm support burst traffic?" |
| Specific version or date in question | "what changed in OpenTelemetry 1.24.0?" |
| "Why" or "how" questions needing broad context | "why did we choose CRDTs over operational transforms?" |
| Answer likely lives in overview/summary doc | Specific stat buried in a broader stats doc |

```
  Specific: "does the token bucket algorithm support burst traffic above rate limit?"
       ↓  Step-back LLM
  Broader:  "What are API rate limiting algorithms and their trade-offs?"
       ↓  Retrieve
  Doc [5]:  "Token bucket and sliding window counter algorithms..."
             (this broad summary answers the specific question too)
```

In [ ]:
class MockStepbackLLM:
    """
    Simulates a step-back LLM that rephrases hyper-specific queries
    as broader, more general questions.
    """

    _STEPBACKS = {
        'token bucket':          'What are API rate limiting algorithms and their trade-offs?',
        'exponential backoff':   'How do webhook delivery systems handle failures and retries?',
        'rs256':                 'What are JWT security best practices for production environments?',
        'none algorithm':        'What JWT algorithm vulnerabilities exist and how are they prevented?',
        'kafka lag':             'What causes streaming data pipeline performance issues?',
        'crdt':                  'What consistency models exist in distributed database systems?',
        'tail-based':            'What are distributed tracing and observability strategies?',
        'n+1':                   'What causes database query performance problems in applications?',
        'authorization code':    'How does OAuth 2.0 work for application authentication?',
    }

    def stepback(self, query: str) -> str:
        q = query.lower()
        for key, broader in self._STEPBACKS.items():
            if key in q:
                return broader
        # Generic step-back: strip the last specific entity and generalise
        words = query.split()
        return f'What are the general principles and approaches for: {" ".join(words[-6:])}?'


stepback_llm = MockStepbackLLM()


def stepback_retrieve(query: str, top_k: int = 3) -> tuple:
    """Broaden the query, retrieve on the broader version. Returns (results, broader_query)."""
    broader = stepback_llm.stepback(query)
    results = retrieve(broader, top_k=top_k)
    return results, broader


# Demo: specific technical queries that benefit from step-back
specific_queries = [
    ('does the token bucket algorithm provide burst tolerance above the rate limit?', 5),
    ('does the none algorithm in JWT pose a security vulnerability?',                4),
    ('does tail-based sampling retain traces only when anomalies occur?',            9),
]

print('=== Step-back Prompting Demo ===\n')
for query, relevant_id in specific_queries:
    base_results           = retrieve(query, top_k=3)
    sb_results, broader    = stepback_retrieve(query, top_k=3)

    base_rank = next((i+1 for i, (d, _) in enumerate(base_results) if d['id'] == relevant_id), 'miss')
    sb_rank   = next((i+1 for i, (d, _) in enumerate(sb_results)   if d['id'] == relevant_id), 'miss')

    base_score = next((s for d, s in base_results if d['id'] == relevant_id), 0.0)
    sb_score   = next((s for d, s in sb_results   if d['id'] == relevant_id), 0.0)

    print(f'Specific:  "{query}"')
    print(f'Broader:   "{broader}"')
    print(f'Relevant:  [{relevant_id}] {CORPUS[relevant_id]["title"]}')
    print(f'Baseline → rank #{base_rank}  (sim={base_score:.3f})')
    print(f'Step-back → rank #{sb_rank}  (sim={sb_score:.3f})')
    print()

---
## 6. Decision Logic: When to Use Which Strategy

Rewriting every query is wasteful — the extra LLM call adds latency and cost. Route to the right strategy based on query characteristics.

```
┌────────────────────────────────────────────────────────────────────────┐
│                     QUERY ROUTING DECISION TREE                        │
└────────────────────────────────────────────────────────────────────────┘

  Incoming query
       │
       ├─── Contains multiple distinct questions? ("and how", "also", ";") ──▶ DECOMPOSE
       │
       ├─── Contains very specific term, algorithm, or named entity? ─────────▶ STEP-BACK
       │    ("token bucket", "RS256", "CRDT", "tail-based sampling")
       │
       ├─── Short, casual, symptom-oriented? (≤ 10 words, no tech jargon) ────▶ HyDE
       │    ("why is my app slow?", "my webhooks keep failing")
       │
       └─── Well-formed, technical, single-hop ─────────────────────────────▶ NO REWRITE
            ("What is the difference between RS256 and ES256?")

  In practice: use a cheap classifier (small LLM or keyword rules) so the
  routing decision costs far less than a full rewrite.
```

In [ ]:
# Specific terms that indicate a step-back is needed
_SPECIFIC_TERMS = {
    'token bucket', 'sliding window', 'exponential backoff', 'rs256', 'es256',
    'none algorithm', 'crdt', 'vector clock', 'tail-based', 'head-based',
    'n+1', 'pg_stat_statements', 'authorization code', 'kafka lag',
}
# Terms that indicate the query is already technical (don't HyDE)
_TECH_JARGON = {
    'oauth', 'jwt', 'api', 'database', 'cache', 'http', 'endpoint',
    'algorithm', 'latency', 'throughput', 'replica', 'distributed',
}


def classify_query(query: str) -> str:
    """
    Route a query to the right rewriting strategy.

    Returns: 'decompose' | 'hyde' | 'stepback' | 'none'
    """
    q     = query.lower().strip()
    words = q.split()

    # 1. Multi-question → decompose
    multi_indicators = ['and how', 'and what', 'and why', 'and our', 'also', ' ; ']
    if any(indicator in q for indicator in multi_indicators):
        return 'decompose'

    # 2. Hyper-specific term → step-back
    if any(term in q for term in _SPECIFIC_TERMS):
        return 'stepback'

    # 3. Short casual query (symptom-oriented) → HyDE
    is_casual = len(words) <= 12 and not any(jargon in q for jargon in _TECH_JARGON)
    if is_casual:
        return 'hyde'

    # 4. Well-formed technical query → no rewriting needed
    return 'none'


def smart_retrieve(query: str, top_k: int = 3, verbose: bool = True) -> dict:
    """
    Complete pipeline: classify query → select strategy → rewrite → retrieve.
    """
    strategy = classify_query(query)

    if strategy == 'decompose':
        results, sub_qs = decompose_retrieve(query, top_k_per_sub=2)
        results         = results[:top_k]
        rewrite_info    = f'Decomposed into {len(sub_qs)} sub-queries'

    elif strategy == 'hyde':
        results, hyp    = hyde_retrieve(query, top_k=top_k)
        rewrite_info    = f'HyDE: "{hyp[:70]}..."'

    elif strategy == 'stepback':
        results, broader = stepback_retrieve(query, top_k=top_k)
        rewrite_info     = f'Step-back: "{broader}"'

    else:
        results      = retrieve(query, top_k=top_k)
        rewrite_info = 'No rewriting (query already well-formed)'

    if verbose:
        print(f'Q: "{query}"')
        print(f'   Strategy: {strategy.upper()} | {rewrite_info}')
        for i, (doc, score) in enumerate(results, 1):
            print(f'   [{i}] [{doc["id"]}] {doc["title"]} (score={score:.3f})')
        print()

    return {'query': query, 'strategy': strategy, 'results': results}


# Classify a range of queries to verify routing
routing_demo = [
    'why is my app so slow?',
    'my server keeps running out of memory',
    'my app is slow and our auth login keeps getting bypassed',
    'does the token bucket algorithm support burst traffic?',
    'does tail-based sampling retain anomaly traces?',
    'how does the OAuth 2.0 authorization code flow work?',
]

print(f'{"Query":<60} {"Strategy"}')
print('-' * 80)
for q in routing_demo:
    strategy = classify_query(q)
    print(f'{q:<60} {strategy.upper()}')

In [ ]:
# Run the combined pipeline on a diverse set of queries
pipeline_queries = [
    'why is my app so slow?',
    'my server keeps running out of memory',
    'my app is slow under load and our auth keeps getting bypassed',
    'does the token bucket algorithm support burst traffic above the rate limit?',
    'does the none algorithm in JWT pose a security risk?',
    'how does the OAuth 2.0 authorization code grant flow work?',
]

print('=== Smart Retrieve Pipeline Demo ===\n')
for q in pipeline_queries:
    smart_retrieve(q, top_k=3)

---
## 7. Recall@k Evaluation

**Recall@k** = 1 if the relevant document appears in the top-k results, 0 otherwise.

We run this on 10 casual queries against the formal technical corpus, comparing:
- **Baseline**: raw query directly to vector DB
- **HyDE**: hypothetical answer as search vector
- **Smart retrieve**: classifier picks the right strategy per query

In [ ]:
# 10 casual queries — each maps to one relevant technical document
EVAL_SET = [
    {'query': 'why is my app so slow?',                                    'relevant_id': 0},
    {'query': 'my server keeps running out of memory',                     'relevant_id': 1},
    {'query': 'how do i make my database faster for lots of reads',        'relevant_id': 2},
    {'query': 'how do i add google login to my app?',                      'relevant_id': 3},
    {'query': 'are my login tokens safe from attacks?',                    'relevant_id': 4},
    {'query': 'my api keeps getting throttled after too many calls',       'relevant_id': 5},
    {'query': 'my webhooks are failing and dropping messages',             'relevant_id': 6},
    {'query': 'my real-time data pipeline keeps falling behind',           'relevant_id': 7},
    {'query': 'my database shows different data on different servers',     'relevant_id': 8},
    {'query': 'how do i see what my microservices are doing',              'relevant_id': 9},
]


def recall_at_k(results: list, relevant_id: int, k: int = 3) -> int:
    """1 if relevant doc is in top-k results, 0 otherwise."""
    top_ids = [doc['id'] for doc, _ in results[:k]]
    return 1 if relevant_id in top_ids else 0


# Run evaluation
baseline_hits = []
hyde_hits     = []
smart_hits    = []

K = 3
print(f'Recall@{K} Evaluation — 10 casual queries vs. formal technical corpus\n')
print(f'{"Query":<55} {"Baseline":<10} {"HyDE":<10} {"Smart":<10} {"Strategy"}')
print('-' * 100)

for item in EVAL_SET:
    q   = item['query']
    rid = item['relevant_id']

    base_results             = retrieve(q, top_k=K)
    hyde_results, _          = hyde_retrieve(q, top_k=K)
    smart_result             = smart_retrieve(q, top_k=K, verbose=False)

    b_hit = recall_at_k(base_results, rid, K)
    h_hit = recall_at_k(hyde_results, rid, K)
    s_hit = recall_at_k(smart_result['results'], rid, K)

    baseline_hits.append(b_hit)
    hyde_hits.append(h_hit)
    smart_hits.append(s_hit)

    marks = lambda x: '✅' if x else '❌'
    strat = smart_result['strategy'].upper()
    print(f'{q:<55} {marks(b_hit):<10} {marks(h_hit):<10} {marks(s_hit):<10} {strat}')

print()
print(f'Recall@{K}  Baseline: {sum(baseline_hits)}/{len(EVAL_SET)} = {np.mean(baseline_hits):.0%}')
print(f'Recall@{K}  HyDE:     {sum(hyde_hits)}/{len(EVAL_SET)} = {np.mean(hyde_hits):.0%}')
print(f'Recall@{K}  Smart:    {sum(smart_hits)}/{len(EVAL_SET)} = {np.mean(smart_hits):.0%}')

In [ ]:
# Visualise Recall@3 comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: per-query hit/miss
x      = np.arange(len(EVAL_SET))
width  = 0.26
labels = [f'Q{i+1}' for i in range(len(EVAL_SET))]

axes[0].bar(x - width, baseline_hits, width, color='#E53935', alpha=0.8, label='Baseline')
axes[0].bar(x,         hyde_hits,     width, color='#F57F17', alpha=0.8, label='HyDE')
axes[0].bar(x + width, smart_hits,    width, color='#2E7D32', alpha=0.8, label='Smart retrieve')
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels, fontsize=10)
axes[0].set_yticks([0, 1])
axes[0].set_yticklabels(['Miss', 'Hit'], fontsize=10)
axes[0].set_title(f'Recall@{K} per Query: Baseline vs. HyDE vs. Smart', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].set_ylim(-0.1, 1.3)

# Right: aggregate recall bars
strategies = ['Baseline\n(raw query)', 'HyDE\n(hypothetical)', 'Smart retrieve\n(routed strategy)']
recalls    = [np.mean(baseline_hits), np.mean(hyde_hits), np.mean(smart_hits)]
colors     = ['#E53935', '#F57F17', '#2E7D32']

bars = axes[1].bar(strategies, recalls, color=colors, alpha=0.85, width=0.5)
axes[1].set_ylim(0, 1.15)
axes[1].set_ylabel(f'Recall@{K}')
axes[1].set_title(f'Aggregate Recall@{K}\n10 casual queries vs. formal docs', fontweight='bold')
axes[1].yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(xmax=1))

for bar, val, hits in zip(bars, recalls, [baseline_hits, hyde_hits, smart_hits]):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{sum(hits)}/{len(EVAL_SET)}\n({val:.0%})',
                 ha='center', fontsize=11, fontweight='bold')

show_plot()

print('What the numbers mean:')
print(f'  Baseline → HyDE improvement:   +{(np.mean(hyde_hits) - np.mean(baseline_hits)):.0%} absolute Recall@3')
print(f'  Baseline → Smart improvement:  +{(np.mean(smart_hits) - np.mean(baseline_hits)):.0%} absolute Recall@3')
print()
print('In production, the improvement depends on:')
print('  • How large the vocabulary gap is in your specific domain')
print('  • How good your HyDE prompt is at generating realistic doc-like text')
print('  • How well your classifier routes to the right strategy')

In [ ]:
# Trade-off analysis: rewriting adds an LLM call — is it worth it?
# This cell frames the cost-benefit decision.

print('=== Trade-off Analysis: Cost vs. Quality ===\n')

tradeoffs = [
    ('Baseline (no rewriting)',    'None',    '0 extra', '0 extra', np.mean(baseline_hits)),
    ('HyDE',                       '1 LLM call per query',    '~500 tokens', '~50-200ms', np.mean(hyde_hits)),
    ('Step-back',                  '1 LLM call per query',    '~100 tokens', '~50-200ms', None),
    ('Decompose (3 sub-queries)',   '1 LLM + 3x retrieval',   '~300 tokens', '~150-600ms', None),
    ('Smart (routed)',             '0-1 LLM per query',       'Varies',       'Varies',     np.mean(smart_hits)),
]

print(f'{"Strategy":<30} {"LLM calls":<25} {"Extra tokens":<16} {"Latency add":<16} {"Recall@3"}')
print('-' * 105)
for name, llm, tokens, latency, recall in tradeoffs:
    r_str = f'{recall:.0%}' if recall is not None else 'n/a'
    print(f'{name:<30} {llm:<25} {tokens:<16} {latency:<16} {r_str}')

print()
print('Key guidance:')
print('  1. Don\'t rewrite every query. Use a cheap classifier first.')
print('  2. HyDE is the best default for casual-query / formal-doc mismatch.')
print('  3. Decompose only when the query is genuinely multi-hop.')
print('  4. Step-back only when a specific entity drowns out the broader topic.')
print('  5. A bad rewrite can make a good query worse — monitor post-deployment.')

---
## 8. Using a Real LLM (Claude API)

In production, each rewriting call goes to a real LLM. Below is the drop-in replacement using Claude — note that a small, fast model (Haiku) is appropriate here because rewriting is a simple instruction-following task, not reasoning.

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`

In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass


DECOMPOSE_PROMPT = """Break this question into 2-4 simpler sub-questions that can each
be answered independently. Output only the sub-questions, one per line.

Question: {query}
Sub-questions:"""

HYDE_PROMPT = """Write a short, plausible paragraph (3-4 sentences) that answers this
question as if you were the relevant technical documentation. Use formal, specific
technical language. It is okay if some details are imprecise.

Question: {query}
Answer:"""

STEPBACK_PROMPT = """Rephrase this specific question as a broader, more general question
that would retrieve a document containing the answer as a specific detail.
Output only the rephrased question.

Specific: {query}
Broader:"""


if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_rewrite(query: str, strategy: str) -> list:
        """Use Claude to rewrite the query according to the chosen strategy."""
        if strategy == 'decompose':
            prompt = DECOMPOSE_PROMPT.format(query=query)
        elif strategy == 'hyde':
            prompt = HYDE_PROMPT.format(query=query)
        elif strategy == 'stepback':
            prompt = STEPBACK_PROMPT.format(query=query)
        else:
            return [query]  # no rewriting

        response = client.messages.create(
            model='claude-haiku-4-5-20251001',  # fast + cheap — rewriting is simple
            max_tokens=256,
            messages=[{'role': 'user', 'content': prompt}],
        )
        text = response.content[0].text.strip()

        if strategy == 'decompose':
            return [line.strip().lstrip('0123456789.-) ') for line in text.splitlines() if line.strip()]
        return [text]

    def claude_smart_retrieve(query: str, top_k: int = 3, verbose: bool = True) -> dict:
        """Full pipeline with Claude as the rewriter."""
        strategy  = classify_query(query)
        rewritten = claude_rewrite(query, strategy)

        if strategy == 'decompose':
            merged = {}
            for sq in rewritten:
                for doc, score in retrieve(sq, top_k=2):
                    if doc['id'] not in merged or score > merged[doc['id']][1]:
                        merged[doc['id']] = (doc, score)
            results = sorted(merged.values(), key=lambda x: x[1], reverse=True)[:top_k]
        else:
            # For HyDE and step-back, rewritten[0] is the new search text
            q_emb   = embedder.encode(rewritten[0], convert_to_tensor=True, show_progress_bar=False)
            scores  = util.cos_sim(q_emb, corpus_embeds)[0].cpu().numpy()
            indices = np.argsort(scores)[::-1][:top_k]
            results = [(CORPUS[i], float(scores[i])) for i in indices]

        if verbose:
            print(f'Q:        "{query}"')
            print(f'Strategy: {strategy.upper()}')
            for i, r in enumerate(rewritten, 1):
                print(f'Rewrite {i}: "{r[:90]}"')
            for i, (doc, score) in enumerate(results, 1):
                print(f'  [{i}] [{doc["id"]}] {doc["title"]} ({score:.3f})')
            print()
        return {'query': query, 'strategy': strategy, 'results': results}

    # Run on a few demo queries
    claude_test = [
        'why is my app so slow?',
        'my app is slow and our login keeps getting bypassed',
        'does the token bucket algorithm provide burst tolerance?',
    ]
    print('=== Claude-powered query rewriting ===')
    for q in claude_test:
        claude_smart_retrieve(q)

    # Recall@3 with Claude
    print('\n=== Recall@3 with Claude rewriting ===')
    claude_hits = []
    for item in EVAL_SET:
        result = claude_smart_retrieve(item['query'], verbose=False)
        claude_hits.append(recall_at_k(result['results'], item['relevant_id'], K))
    print(f'Claude Recall@{K}: {sum(claude_hits)}/{len(EVAL_SET)} = {np.mean(claude_hits):.0%}')
    print(f'Mock   Recall@{K}: {sum(smart_hits)}/{len(EVAL_SET)} = {np.mean(smart_hits):.0%}')

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable this section:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('The prompts above show exactly what to send to any LLM API.')
    print()
    print('Model sizing guidance for the rewriter:')
    print('  HyDE / step-back:  claude-haiku-4-5-20251001  (cheap, fast — simple task)')
    print('  Decomposition:     claude-haiku-4-5-20251001  (fast enough for planning)')
    print('  Complex planning:  claude-sonnet-4-6          (if sub-query quality matters)')
    print()
    print('The rewriter does not need to be the same model as the generator.')
    print('Using a small fast model for rewriting saves cost and latency.')

---
## Key Takeaways

1. **The query–document mismatch is the biggest silent killer of RAG quality.** Users write casual, symptom-oriented queries; documents use formal, cause-oriented language. They share almost no vocabulary, so their embeddings land far apart in vector space.

2. **Don't search with the raw query — rewrite it first.** The rewriter sits between the user and the retriever, acting as a translator between "human" and "document" language.

3. **HyDE is the most broadly useful technique.** Generate a hypothetical formal answer (even a wrong one), embed that instead of the query. The answer uses the same vocabulary as real documents and closes the embedding gap — as shown by the cosine similarity comparison and PCA plot.

4. **Sub-query decomposition handles compound questions.** Complex multi-hop questions cannot be answered in a single retrieval call. Decompose → retrieve per sub-query → merge with max-score deduplication.

5. **Step-back prompting handles hyper-specific queries.** When the answer lives in a broader summary document, zooming out before retrieval beats searching for the exact entity.

6. **Route selectively — not every query needs rewriting.** A cheap classifier (keyword rules or a small LLM) decides which strategy to apply. Rewriting every query adds unnecessary latency and cost, and a bad rewrite can make a good query worse.

7. **Use a small, fast model for rewriting.** Rewriting is a simple instruction-following task. `claude-haiku-4-5-20251001` or an equivalent fast model adds minimal latency while providing real LLM quality — far better than mock keyword matching.

---

*Up next: Lesson 8.3 — "Once your queries are clean, how do you actually fetch documents for complex questions?" — iterative and multi-step retrieval: the retriever doesn't just run once. It runs in a loop, using each result to decide what to retrieve next.*